In [1]:
import pandas as pd
import numpy as np
import ipyparallel as ipp

In [2]:
def ranked_probability_scores(observations, predictions):
    
    N = len(observations)
    rps = np.zeros(N)
    
    for i in range(N):
        
        observation_prob = np.array([0,0,0])
        observation_prob[observations[i]] = 1
            
        rps[i] = 1/2*( (np.cumsum(predictions[i])-np.cumsum(observation_prob))**2 ).sum()

    return rps

In [3]:
def validation_step(df_X, df_y, winter, start_winter, max_tree_depth, min_leaf_size, split_crit, n_trees, chunksize=1):
    
    is_validation_data = (winter >= start_winter) & (winter < start_winter + chunksize)
    X_valid = df_X[is_validation_data].values
    X_train = df_X[~is_validation_data].values
    y_valid = df_y[is_validation_data].values[:,0]
    y_train = df_y[~is_validation_data].values[:,0]
        
    classifier = RandomForestClassifier(n_estimators=n_trees, criterion=split_crit, max_depth=max_tree_depth, 
                                        min_samples_leaf=min_leaf_size, random_state=42)
    
    classifier.fit(X_train, y_train)
    y_pred = classifier.predict_proba(X_valid)
    rps = ranked_probability_scores(y_valid, y_pred)
    
    return rps.mean()

In [26]:
df_features = pd.read_csv('features/features_NE_week5.csv')
df_target = pd.read_csv('targets/northern_temperature_terc_cat_new.csv')

In [27]:
df_features['time'] = pd.to_datetime(df_features['time'])
df_features = df_features.set_index('time')

df_target['time'] = pd.to_datetime(df_target['time'])
df_target = df_target.set_index('time')

In [28]:
df_target_train = df_target[df_target.index < "2010-07"]

In [29]:
df_features.columns

Index(['MJO_phase', 'nino3.4_index', 'nao_index', 'sea_ice_arctic',
       'sea_ice_baka', 'soil_moist_upper', 'soil_moist_lower', 'snow_depth',
       'u10_60N'],
      dtype='str')

In [36]:
drop = ['snow_depth', 'nino3.4_index', 'sea_ice_arctic', 'soil_moist_upper', 'sea_ice_baka', 'MJO_phase', 'soil_moist_lower', 'nao_index']

In [37]:
df_features_sel = df_features.drop(columns=drop)
df_features_train = df_features_sel[df_features_sel.index < "2010-07"]
len(df_features_train.columns)

1

In [10]:
n = 60
cluster = ipp.Cluster(n = n)
c = cluster.start_and_connect_sync()

Starting 60 engines with <class 'ipyparallel.cluster.launcher.LocalEngineSetLauncher'>
100%|██████████| 60/60 [00:06<00:00,  9.34engine/s]


In [11]:
dview = c[:]
dview.block = True 

In [12]:
dview.execute('from sklearn.ensemble import RandomForestClassifier')
dview.execute('import numpy as np')

<AsyncResult(execute): finished>

In [32]:
N = 1000
d_max = None
min_samples_leaf = 1200
criterion = 'gini'

In [33]:
year = df_features_train.index.year
month = df_features_train.index.month
winter = year.where(month < 12, year+1)
start_winters = np.arange(winter[0], winter[-1]+1, 1)

In [38]:
mydict = dict(df_features_train=df_features_train, df_target_train=df_target_train, validation_step=validation_step, ranked_probability_scores=ranked_probability_scores,
              N=N, d_max=d_max, min_samples_leaf=min_samples_leaf, criterion=criterion, winter=winter, start_winters=start_winters)
dview.push(mydict)

[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None]

In [16]:
lview = c.load_balanced_view()
lview.block = True

In [39]:
%%time
def wrapper(i):    
    return(validation_step(df_features_train, df_target_train, winter, start_winters[i], d_max, min_samples_leaf, criterion, N))

result_list = lview.map(wrapper, range(n))
np.array(result_list).mean()

CPU times: user 92.7 ms, sys: 12.9 ms, total: 106 ms
Wall time: 2.79 s


np.float64(0.22346116034006683)